In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
data = [ 
(101, "C001", "2026-04-01", "P101", 2, 500, "Delhi", "Completed"), 
(102, "C002", "2026-04-01", "P102", 1, 1200, "Mumbai", "Completed"), 
(103, "C001", "2026-04-02", "P103", 3, 300, "Delhi", "Cancelled"), 
(104, "C003", "2026-04-02", "P101", 1, 500, "Pune", "Completed"), 
(105, "C004", "2026-04-03", "P104", 4, 150, "Bangalore", "Completed"), 
(106, "C002", "2026-04-03", "P109", 2, 1200, "Mumbai", "Returned"), 
(107, "C005", "2026-04-04", "P105", 1, None, "Chennai", "Completed"), 
(108, "C006", "2026-04-04", "P106", None, 800, "Hyderabad", "Completed") 
] 

 

# "columns = [order_id, customer_id, order_date, product_id, quantity, price, city, order_status ]"
columns = ["order_id", "customer_id", "order_date", "product_id", "quantity", "price", "city", "order_status"] 
order_df = spark.createDataFrame(data,columns)
order_df.display()




In [0]:
data  = [ 
("C001", "Aseem", "Premium"), 
("C002", "Ravi", "Gold"), 
("C003", "Neha", "Silver"), 
("C004", "Priya", "Gold"), 
("C005", "Arjun", "Premium"), 
("C006", "Kiran", "Silver") 
]

columns = ["customer_id", "customer_name", "segment"]

customer_df = spark.createDataFrame(data,columns)
customer_df.display()

In [0]:
data = [ 
("P101", "Laptop Bag", "Accessories"), 
("P102", "Mobile", "Electronics"), 
("P103", "Keyboard", "Electronics"), 
("P104", "Notebook", "Stationery"), 
("P105", "Chair", "Furniture"), 
("P106", "Monitor", "Electronics") 
] 

columns = ["product_id", "product_name", "category"]

product_df = spark.createDataFrame(data,columns)
product_df.display()


In [0]:
data = [ 
("C001", ["aseem@email.com","aseem.work@email.com"]), 
("C002", ["ravi@email.com"]), 
("C003", ["neha@email.com","neha.office@email.com"]), 
("C004", ["priya@email.com"]) 
] 
columns = ["customer_id", "email"]

customer_contact_df = spark.createDataFrame(data,columns)
customer_contact_df.display()

In [0]:
#Null Values 
avg_price = order_df.groupBy().avg("price").collect()[0][0]

order_df = order_df.withColumn("price", when(col("price").isNull(),avg_price).otherwise(col("price"))).withColumn("quantity",when(col("quantity").isNull(),lit(1)).otherwise(col("quantity")))
order_df.display()

In [0]:
schema = StructType([
    StructField("customer_id", StringType()),
    StructField("event_time",  StringType()),
    StructField("event_type",  StringType()),
    StructField("amount",      IntegerType()),
])

data = [
    ("C1", "2024-01-01 09:00", "view",        None),
    ("C1", "2024-01-01 09:10", "add_to_cart", None),
    ("C1", "2024-01-01 09:30", "purchase",    100),
    ("C1", "2024-01-02 11:00", "view",        None),
    ("C1", "2024-01-02 11:30", "purchase",    250),
    ("C2", "2024-01-03 10:00", "view",        None),
    ("C2", "2024-01-03 10:20", "purchase",    300),
    ("C3", "2024-01-04 12:00", "view",        None),
    ("C3", "2024-01-04 12:10", "add_to_cart", None),
    ("C3", "2024-01-04 12:20", "view",        None),
    ("C4", "2024-01-05 08:00", "view",        None),
    ("C4", "2024-01-05 08:20", "purchase",    150),
    ("C4", "2024-01-05 09:00", "purchase",    200),
    ("C4", "2024-01-06 10:00", "purchase",    50),
    ("C5", "2024-01-07 15:00", "view",        None),
    ("C5", "2024-01-07 15:30", "view",        None),
    ("C5", "2024-01-07 16:00", "purchase",    80),
]

df = spark.createDataFrame(data, schema)
df.display()


In [0]:
first_purchase_df = df.filter(col("event_type")=="purchase").groupBy("customer_id").agg(min("event_time").alias("First_purchase_date"))
first_purchase_df.display()

In [0]:
df_joined = df.join(first_purchase_df,on = "customer_id",how="left")
df_joined.display()
result = df_joined.filter((col("event_type")=="purchase") & (col("event_time")>col("first_purchase_date")))
result.display()

In [0]:
after_spend = result.groupBy("customer_id", "First_purchase_date") \
                    .agg(sum("amount").alias("total_after_first_purchase"))


In [0]:
after_spend.display()

In [0]:
all_customers = first_purchase_df.join(after_spend, on=["customer_id", "First_purchase_date"], how="left") \
                               .fillna(0, subset=["total_after_first_purchase"])

In [0]:
all_customers.display()

In [0]:
from pyspark.sql.window import Window
window = Window.partitionBy("customer_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)

In [0]:
# ─── DATA ─────────────────────────────────────────────────────
# Orders — Amazon is the HOT KEY (too many rows)
orders_data = [
    # Amazon orders — 10 rows (in real life this would be 900 million)
    ("Amazon", "O_001", 500),
    ("Amazon", "O_002", 300),
    ("Amazon", "O_003", 700),
    ("Amazon", "O_004", 200),
    ("Amazon", "O_005", 900),
    ("Amazon", "O_006", 150),
    ("Amazon", "O_007", 450),
    ("Amazon", "O_008", 600),
    ("Amazon", "O_009", 250),
    ("Amazon", "O_010", 800),
    # Other customers — small
    ("Flipkart", "O_011", 100),
    ("Flipkart", "O_012", 200),
    ("Meesho",   "O_013", 300),
]

# Customers — small table
customers_data = [
    ("Amazon",   "Seattle",  "USA"),
    ("Flipkart", "Bangalore","India"),
    ("Meesho",   "Mumbai",   "India"),
]

orders    = spark.createDataFrame(orders_data,    ["customer_id", "order_id", "amount"])
customers = spark.createDataFrame(customers_data, ["customer_id", "city", "country"])


In [0]:
customers.display()orders


In [0]:
text = input("Enter text: ")
words = text.split()

for word in words:
    reversed_word = word[::-1].lower()
    result = reversed_word + " "


print(result)

In [0]:
str = "    Ahya##an   hello"

print(str.strip("#"))

In [0]:
orders.display()

In [0]:
orders.groupBy("customer_id").count().orderBy(desc("count")).display()

##Amazon is the Hot Key 

In [0]:
normal_join = orders.join(customers,on="customer_id",how="left")
normal_join.display()

###Print Rows Before Salting

In [0]:
normal_join.withColumn("partition_id", spark_partition_id()) \
           .groupBy("partition_id") \
           .count() \
           .orderBy("partition_id")\
               .display()